In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Base

In [ ]:
import pandas as pd
import os


input_file = '/content/drive/MyDrive/Data Sets/Guarani/guarani-train.tsv'
output_file = '/content/drive/MyDrive/Data Sets/Guarani/knowledge_base.txt'

# Asegurar directorio
os.makedirs('data/book/', exist_ok=True)

# Crear un archivo de texto dummy si no tienes el tsv aún para probar
if not os.path.exists(input_file):
    print(f"No encuentro {input_file}. Creando datos de prueba...")
    data = {
        'Source': ['Ore ndorombyai kuri', 'Che akaru', 'Ndaipori problema'],
        'Change': ['TYPE:AFF', 'TYPE:NEG', 'TYPE:FUTURE'],
        'Target': ['Ore rombyai kuri', 'Che ndakarui', 'Ndaiporimoai problema']
    }
    df = pd.DataFrame(data)
else:
    # Cargar el TSV real (separado por tabs)
    df = pd.read_csv(input_file, sep='\t')

# Guardar como texto explicativo para el RAG
with open(output_file, 'w', encoding='utf-8') as f:
    for index, row in df.iterrows():
        # Escribimos en formato de "Regla" para que el modelo lo entienda
        text = f"""
Ejemplo de Transformación Gramatical:
Frase Original (Guaraní): {row['Source']}
Instrucción de Cambio: {row['Change']}
Resultado Correcto: {row['Target']}
---
"""
        f.write(text)

print(f"¡Listo! Base de conocimiento creada en {output_file}")

¡Listo! Base de conocimiento creada en /content/drive/MyDrive/Data Sets/Guarani/knowledge_base.txt


### Ingest

In [ ]:
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Para los loaders, en algunas versiones pueden estar en lugares diferentes:
try:
    # Intenta importar de la nueva ubicación primero
    from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
except ImportError:
    # Fallback a la ubicación antigua
    from langchain.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Tu código continúa aquí...

In [ ]:
DATA_PATH = '/content/drive/MyDrive/Data Sets/Guarani'
DB_FAISS_PATH = '/content/drive/MyDrive/Data Sets/Guarani/db_faiss'

def create_vector_db():
    # Verificar que exista el directorio
    if not os.path.exists(DATA_PATH):
        os.makedirs(DATA_PATH)
        print(f"Directorio {DATA_PATH} creado. Por favor sube tus archivos aquí.")
        return

    print("--- Iniciando Ingesta de Documentos ---")

    # 1. Cargar PDFs (Diccionarios, Gramática)
    print("Cargando PDFs...")
    loader_pdf = DirectoryLoader(DATA_PATH,
                                 glob='*.pdf',
                                 loader_cls=PyPDFLoader)
    try:
        docs_pdf = loader_pdf.load()
        print(f" -> Encontrados {len(docs_pdf)} fragmentos de PDF.")
    except Exception as e:
        print(f" -> No se pudieron cargar PDFs o no hay archivos: {e}")
        docs_pdf = []

    # 2. Cargar TXTs (Dataset de entrenamiento convertido, ejemplos)
    print("Cargando archivos de texto (.txt)...")
    loader_txt = DirectoryLoader(DATA_PATH,
                                 glob='*.txt',
                                 loader_cls=TextLoader)
    try:
        docs_txt = loader_txt.load()
        print(f" -> Encontrados {len(docs_txt)} fragmentos de texto.")
    except Exception as e:
        print(f" -> No se pudieron cargar TXTs o no hay archivos: {e}")
        docs_txt = []

    # 3. Unificar documentos
    documents = docs_pdf + docs_txt

    if not documents:
        print("ERROR: No se encontraron documentos ni PDF ni TXT en 'data/book/'.")
        print("Sube el PDF de gramática o ejecuta el script de conversión de TSV.")
        return

    # 4. Dividir texto (Splitting)
    # Usamos un chunk_size más pequeño (600) para reglas gramaticales precisas
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600,
                                                   chunk_overlap=100)
    texts = text_splitter.split_documents(documents)
    print(f"Total de fragmentos (chunks) a procesar: {len(texts)}")

    # 5. Crear Embeddings
    # Usamos un modelo multilingüe ligero.
    # Opcional: Si tienes GPU, cambia device a 'cuda'
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        model_kwargs={'device': 'cpu'}
    )

    # 6. Crear y guardar base de datos vectorial
    print("Creando base de datos vectorial FAISS...")
    db = FAISS.from_documents(texts, embeddings)
    db.save_local(DB_FAISS_PATH)
    print(f"¡Éxito! Base de conocimiento guardada en '{DB_FAISS_PATH}'")

In [ ]:
create_vector_db()

--- Iniciando Ingesta de Documentos ---
Cargando PDFs...
 -> Encontrados 478 fragmentos de PDF.
Cargando archivos de texto (.txt)...
 -> Encontrados 1 fragmentos de texto.
Total de fragmentos (chunks) a procesar: 1751
Creando base de datos vectorial FAISS...
¡Éxito! Base de conocimiento guardada en '/content/drive/MyDrive/Data Sets/Guarani/db_faiss'


### Model 1

In [ ]:
DATA_PATH = '/content/drive/MyDrive/Data Sets/Guarani'
DB_FAISS_PATH = '/content/drive/MyDrive/Data Sets/Guarani/db_faiss'

In [ ]:
USE_RAG = True

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
import os
import torch
import chainlit as cl
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

In [ ]:
MODEL_ID = "google/gemma-2-2b-it"
DB_FAISS_PATH = '/content/drive/MyDrive/Data Sets/Guarani/db_faiss'

# ==========================================
# 1. CARGA DEL MODELO (VERSIÓN COMPATIBLE CPU/GPU)
# ==========================================
def load_llm():
    print(f"⏳ Cargando modelo: {MODEL_ID}...")

    # NOTA: Hemos eliminado BitsAndBytesConfig para evitar el error.
    # El modelo se cargará en su precisión normal (float32 o bfloat16).

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    # Cargamos el modelo sin configuración de 4-bits
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto", # Usará GPU si hay, o CPU si no hay
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.3,
        return_full_text=False,
    )

    return HuggingFacePipeline(pipeline=pipe)

In [ ]:
def set_custom_prompt():
    """
    Prompt diseñado para AmericasNLP: Source -> Change -> Target
    """
    custom_prompt_template = """
    Eres un experto lingüista especializado en Guaraní y Jopará.
    Tu tarea es transformar la oración dada siguiendo estrictamente la instrucción de cambio.

    Usa el siguiente contexto gramatical o de diccionario para guiarte.
    Si el contexto no ayuda, usa tu conocimiento base.

    CONTEXTO (Reglas/Diccionario):
    {context}

    INSTRUCCIÓN DE USUARIO:
    {question}

    Responde SOLAMENTE con la oración transformada en Guaraní. No des explicaciones.
    RESPUESTA:
    """
    prompt = PromptTemplate(template=custom_prompt_template,
                            input_variables=['context', 'question'])
    return prompt

    def set_custom_prompt():
    custom_prompt_template = """
    ### INSTRUCCIÓN:
    Actúa como un motor de transformación gramatical para Guaraní.
    Tu objetivo es aplicar la etiqueta de cambio (CHANGE) a la oración base (SOURCE).

    NO expliques nada. NO escribas código Python. NO definas reglas.
    Solo devuelve la oración transformada.

    ### CONTEXTO DE GRAMÁTICA (ÚSALO SOLO SI APLICA):
    {context}

    ### EJEMPLOS DE REFERENCIA (Sigue este formato):
    Input: Akaru | TYPE:NEG
    Target: Ndakarui

    Input: Oho | TYPE:FUTURE
    Target: Ohota

    Input: Vaka | TYPE:PLURAL
    Target: Vakakuéra

    ### TU TAREA ACTUAL:
    Input: {question}
    Target:
    """
    prompt = PromptTemplate(template=custom_prompt_template,
                            input_variables=['context', 'question'])
    return prompt

In [ ]:
def create_rag_chain():
    llm = load_llm()
    prompt = set_custom_prompt()

    if USE_RAG:
        print("Cargando Base de Conocimiento (RAG Activo)...")
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            model_kwargs={'device': 'cpu'}
        )
        try:
            db = FAISS.load_local(DB_FAISS_PATH, embeddings, allow_dangerous_deserialization=True)
            retriever = db.as_retriever(search_kwargs={'k': 3}) # Traer las 3 reglas más relevantes

            qa_chain = RetrievalQA.from_chain_type(
                llm=llm,
                chain_type='stuff',
                retriever=retriever,
                return_source_documents=True,
                chain_type_kwargs={'prompt': prompt}
            )
            return qa_chain
        except Exception as e:
            print(f"Error cargando FAISS: {e}. Se ejecutará sin contexto.")

    # Si USE_RAG es False o falla la carga, devolvemos una cadena simple (sin retrieval)
    # Nota: Para simplificar, en modo No-RAG chainlit usará el LLM directo,
    # pero aquí mantenemos la estructura.
    return llm

### Ejecucion

In [ ]:
llm = load_llm()

⏳ Cargando modelo: google/gemma-2-2b-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={'device': 'cpu'}
    )

In [ ]:
db = FAISS.load_local(DB_FAISS_PATH, embeddings, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={'k': 3})


In [ ]:
prompt = set_custom_prompt()
chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type='stuff',
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={'prompt': prompt}
        )

In [ ]:
if USE_RAG:
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={'device': 'cpu'}
    )
    try:
        db = FAISS.load_local(DB_FAISS_PATH, embeddings, allow_dangerous_deserialization=True)
        retriever = db.as_retriever(search_kwargs={'k': 3})
        prompt = set_custom_prompt()

        chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type='stuff',
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={'prompt': prompt}
        )
    except Exception as e:
        chain = llm
else:
    chain = llm

In [ ]:
while True:
    user_input = input("\n📝 Tu entrada: Che aipota so'o | TYPE:NEG")

    if user_input.lower() in ['salir', 'exit', 'quit']:
        print("Apagando...")
        break

    print("Pensando...")

    try:
        if USE_RAG and hasattr(chain, 'invoke'):
            # Respuesta con RAG
            response = chain.invoke(user_input)
            result_text = response['result']
            source_docs = response.get('source_documents', [])

            print(f"\n🗣️ RESPUESTA: {result_text.strip()}")

            if source_docs:
                print("\n🔍 Fuentes utilizadas:")
                for i, doc in enumerate(source_docs):
                    # Imprime los primeros 100 caracteres de la fuente para verificar
                    print(f"   [{i+1}] {doc.page_content[:150].replace(chr(10), ' ')}...")
            else:
                print("\n No se encontraron fuentes relevantes.")

        else:
            # Respuesta directa (Sin RAG)
            prompt_simple = f"Instrucción: {user_input}\nRespuesta en Guaraní:"
            response = chain.invoke(prompt_simple)
            print(f"\nRESPUESTA: {response.strip()}")

    except Exception as e:
        print(f"Error al procesar: {e}")